# FastAPI Basics and Request Pipeline
**Topics:** FastAPI Basics Recap · Dependencies & Injection · Middleware & CORS · Advanced Routing

> Code snippets here are for understanding structure — run the `.py` files in `examples/` with `uvicorn` to see them working.

---
## PART 1 — FastAPI Basics Recap

### What is FastAPI?

FastAPI is a Python web framework for building APIs. Two things make it stand out:
1. **Speed** — one of the fastest Python frameworks (built on Starlette + uvicorn)
2. **Auto docs** — go to `/docs` and FastAPI generates interactive Swagger UI automatically from your code

### Installation
```bash
pip install fastapi uvicorn
```

`uvicorn` is the ASGI server that actually runs the app. FastAPI is the framework, uvicorn is what serves it.

### Running an app
```bash
uvicorn main:app --reload
#       ↑    ↑      ↑
#    file  variable  auto-restart on file change
```

## WSGI vs ASGI

**WSGI (2003)** — synchronous, one request per worker at a time. To handle 100 simultaneous requests, you need 100 workers.

**ASGI (2019)** — asynchronous, supports `async/await`. One worker can handle thousands of concurrent connections — while waiting on DB/IO, it picks up other requests instead of blocking.

```
WSGI worker:   Request1 → wait for DB → respond     (blocked during wait)
               Request2 waits in queue...

ASGI worker:   Request1 → wait for DB...
                              ↓ (while waiting)
               Request2 → handled now
               Request3 → handled now
                              ↓ (DB responds)
               Request1 → respond
```

Django and Flask are WSGI frameworks. FastAPI is ASGI-native.

---

## Flask vs FastAPI vs Django

### One-line summary

- **Django** — full web framework, everything included
- **Flask** — minimal, you add what you need
- **FastAPI** — minimal like Flask, but async and modern

### Side by Side

| | Flask | FastAPI | Django |
|---|---|---|---|
| Type | WSGI (sync) | ASGI (async) | WSGI (sync) |
| Speed | Medium | Very fast | Slow–Medium |
| ORM | No built-in | No built-in | Built-in |
| Admin panel | No | No | Yes |
| Auth | No | No | Yes |
| Auto API docs | No | Yes (`/docs`) | No |
| Data validation | Manual | Automatic (Pydantic) | Manual/Forms |
| Learning curve | Easy | Easy–Medium | Steep |
| Best for | Small apps, prototypes | APIs, ML backends | Full web apps |

### Philosophy

```
Django:   "Here's everything you need — ORM, auth, admin, templates, forms"
           → opinionated, makes decisions for you

Flask:    "Here's almost nothing — add what you need"
           → flexible, you make all decisions

FastAPI:  "Here's almost nothing, but with modern async + auto validation"
           → flexible like Flask but with better developer experience
```

### Same Endpoint, All Three

```python
# Flask — no validation, manual JSON
@app.route("/users/<int:user_id>", methods=["GET"])
def get_user(user_id):
    return jsonify({"id": user_id})
```

```python
# FastAPI — auto validates user_id is int, auto JSON serialization
@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"id": user_id}
```

```python
# Django — separate urls.py + views.py
def get_user(request, user_id):
    return JsonResponse({"id": user_id})
```

### When to Pick Which

| Pick | When |
|------|------|
| **Django** | Full product, need admin dashboard, content-heavy site |
| **Flask** | Small app, prototype, max flexibility, lightweight microservice |
| **FastAPI** | API backend for React/mobile, ML model serving, real-time (WebSockets), performance matters |

> For AI/ML engineering — ML APIs, RAG backends, and agent systems are almost always built with FastAPI.

### 1.1 — Creating an App & Basic Routes

In [ ]:
from fastapi import FastAPI

# FastAPI() creates the application instance
# convention is to name it 'app' — same reason numpy is 'np', pandas is 'pd'
# whatever name you use here, use the same in decorators AND in uvicorn command
# e.g. if you name it 'sapp' → @sapp.get() and uvicorn main:sapp --reload
app = FastAPI()

# Route = URL path + HTTP method + function
# decorator @app.get("/") registers this function as the handler for GET /

@app.get("/")
def root():
    return {"message": "Hello"}          # FastAPI auto-converts dict → JSON response

# POST — used for creating resources, data comes in request body
@app.post("/items")
def create_item():
    return {"created": True}

# PUT — used for updating a resource, {id} is a path parameter
# FastAPI reads 'id' from the URL and passes it to the function
# type hint 'int' means FastAPI validates and converts it automatically
# GET /items/abc → 422 error (not an int)
# GET /items/123 → id = 123 (int)
@app.put("/items/{id}")
def update_item(id: int):
    return {"updated": id}

# DELETE — used for deleting a resource
@app.delete("/items/{id}")
def delete_item(id: int):
    return {"deleted": id}

### 1.2 — Path Parameters vs Query Parameters

**Path parameter** — part of the URL itself: `/users/42`  
**Query parameter** — comes after `?`: `/users?age=25&active=true`

In [ ]:
@app.get("/users/{user_id}")        # /users/42
def get_user(user_id: int):         # FastAPI reads 42 from URL, converts to int
    return {"user_id": user_id}


@app.get("/search")                 # /search?q=python&limit=10
def search(
    q: str,                         # required query param
    # optional has default values
    limit: int = 10,                # optional, defaults to 10
    active: bool = True             # optional, defaults to True
):
    return {"query": q, "limit": limit, "active": active}


# Mixed: path + query
@app.get("/users/{user_id}/posts")  # /users/42/posts?page=2
def get_user_posts(user_id: int, page: int = 1):
    return {"user": user_id, "page": page}

## 🔀 FastAPI — Path Params vs Query Params

FastAPI automatically determines parameter types using **one simple rule**:

> If the parameter name appears inside `{}` in the route URL → **path param**
> If it does NOT appear in the URL → **query param**

---

### 📌 Example

```python
@app.get("/users/{user_id}")   # user_id is in the path → path param
def get_user(user_id: int):
    ...

@app.get("/search")            # nothing in the path
def search(
    q: str,                    # required query param   (no default)
    limit: int = 10,           # optional query param   (has default)
    active: bool = True        # optional query param   (has default)
):
    ...
```

---

### ✅ Optional vs Required

| Rule | Type |
|------|------|
| No default value | **Required** — must be passed |
| Has a default `= value` | **Optional** — falls back to default |

---

### 🧪 Valid & Invalid Calls

```
GET /search?q=python              ✅  limit=10, active=True (defaults used)
GET /search?q=python&limit=5      ✅  only limit overridden
GET /search                       ❌  422 Unprocessable Entity — q is required
```

---

> 💡 **Tip:** FastAPI reads your function signature + the URL pattern — no decorators or extra config needed. Just position and defaults do all the work.

### 1.3 — Request Body with Pydantic

For POST/PUT requests, data comes in the **request body** as JSON.  
Pydantic models define the shape — FastAPI validates and parses automatically.

In [ ]:
from pydantic import BaseModel
from typing import Optional

# Define the shape of incoming JSON
class Item(BaseModel):
    name: str
    price: float
    description: Optional[str] = None   # optional field
    in_stock: bool = True               # optional with default


@app.post("/items")
def create_item(item: Item):            # FastAPI reads JSON body → Item object
    return {
        "name": item.name,
        "total": item.price * 1.1       # access fields directly
    }

# Request JSON:  {"name": "Laptop", "price": 999.99}
# Response:      {"name": "Laptop", "total": 1099.989}

### 1.4 — Response Models & Status Codes

In [ ]:
from fastapi import HTTPException, status

class ItemResponse(BaseModel):
    id: int
    name: str
    # price is NOT here — FastAPI will strip it from response


@app.post(
    "/items",
    response_model=ItemResponse,        # defines what the response looks like
    status_code=status.HTTP_201_CREATED # returns 201 instead of default 200
)
def create_item(item: Item):
    # even if you return price, FastAPI strips it because it's not in ItemResponse
    return {"id": 1, "name": item.name, "price": item.price}

# Raising errors
@app.get("/items/{item_id}")
def get_item(item_id: int):
    if item_id > 100:
        raise HTTPException(
            status_code=404,
            detail=f"Item {item_id} not found"
        )
    return {"item_id": item_id}

---
## PART 2 — Dependencies & Injection

In FastAPI, dependencies are reusable functions that run before your endpoint and inject their result into it.

This concept exists in other backend frameworks too.
The concept is the same everywhere, but the implementation differs.

Every backend framework has a way to share reusable logic across routes — they just call it different things and implement it differently.



### The Problem Dependencies Solve

Imagine 10 endpoints, all need to:
- verify a JWT token
- get a database connection
- check if the user is an admin

Without dependencies you'd copy that logic into every function. Dependencies let you write it **once** and inject it wherever needed.

**Real-world analogy:** Like a restaurant — every table needs plates, cutlery, glasses. The kitchen doesn't prepare these from scratch per table. There's a setup system that delivers them. `Depends()` is that system.

### How `Depends()` Works

```
Request comes in
      ↓
FastAPI sees Depends(get_db)
      ↓
Calls get_db() first
      ↓
Passes result into your endpoint function
      ↓
Your function runs
```

### 2.1 — Simple Dependency

In [ ]:
from fastapi import Depends

# A dependency is just a regular function
def get_query_params(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}


# Inject it with Depends() — FastAPI calls get_query_params and passes the result
@app.get("/users")
def list_users(params: dict = Depends(get_query_params)):
    return {"users": [], "pagination": params}


# Same dependency reused in multiple endpoints — write once, use everywhere
@app.get("/products")
def list_products(params: dict = Depends(get_query_params)):
    return {"products": [], "pagination": params}

### 2.2 — Auth Dependency (most common real-world use)

In [ ]:
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi import Depends, HTTPException

# HTTPBearer() reads the Authorization header from the request
# it expects the format:  Authorization: Bearer <token>
# this is itself a dependency — FastAPI calls it automatically
security = HTTPBearer()


def get_current_user(
    # HTTPBearer (security) runs first, extracts the Authorization header
    # and passes it here as HTTPAuthorizationCredentials object
    # which has one useful attribute: .credentials — the raw token string
    credentials: HTTPAuthorizationCredentials = Depends(security)
):
    # extract just the token string from the credentials object
    # if header was "Authorization: Bearer abc123" → credentials.credentials = "abc123"
    token = credentials.credentials

    # validate the token — in reality you'd decode a JWT here using python-jose
    # and check expiry, signature, user existence etc.
    if token != "valid-token":
        raise HTTPException(status_code=401, detail="Invalid token")

    # return the user data — this is what gets injected into the endpoint
    # in reality you'd fetch the user from DB using the decoded token's user_id
    return {"user_id": 1, "email": "user@example.com"}


# full chain when a request hits /profile:
# 1. HTTPBearer() reads Authorization header → extracts token
# 2. get_current_user() validates token → returns user dict
# 3. get_profile() receives user dict as current_user → runs
# if step 1 or 2 raises HTTPException, step 3 never runs

@app.get("/profile")
def get_profile(current_user: dict = Depends(get_current_user)):
    return {"profile": current_user}


@app.get("/dashboard")
def get_dashboard(current_user: dict = Depends(get_current_user)):
    # current_user is the dict returned by get_current_user
    # {"user_id": 1, "email": "user@example.com"}
    return {"user": current_user["email"], "data": []}


### 2.3 — Database Dependency with `yield`

#### `yield` in FastAPI Dependencies — Setup & Cleanup

A `yield` dependency splits into two phases:

```python
def get_db():
    db = FakeDB()   # ① setup   — runs before the endpoint
    yield db        # ② pause   — db gets injected into the endpoint
    db.close()      # ③ cleanup — runs after the endpoint finishes
```

Think of it as a **context manager you don't have to write yourself** —
same idea as `with open("file.txt") as f:`, just FastAPI handles the
`with` part automatically.

---

### ❌ Without `yield` — connection leaks

```python
def get_db():
    db = FakeDB()
    return db       # function ends here — db.close() never runs
```

### ✅ With `yield` — clean lifecycle

```
① open connection
② endpoint runs and uses db
③ db.close() — guaranteed, even if the endpoint crashes
```

---

> 💡 One-liner: `yield` = *"give this value to the endpoint now,
> but come back to me after it finishes so I can clean up."*

In [ ]:
# Simulated DB session
class FakeDB:
    def query(self, model): return []
    def close(self): pass


def get_db():
    db = FakeDB()    # open connection BEFORE the endpoint runs
    try:
        yield db     # pass db into the endpoint
    finally:
        db.close()   # close connection AFTER the endpoint finishes (even on error)


@app.get("/users")
def list_users(db = Depends(get_db)):
    users = db.query("users")    # db is already open, ready to use
    return {"users": users}

### 2.4 — Chained Dependencies

Dependencies can depend on other dependencies — FastAPI resolves the chain automatically.

In [ ]:
def get_current_user(token: str = Depends(security)):
    return {"user_id": 1, "role": "user"}

# This depends on get_current_user, which depends on security
def require_admin(user: dict = Depends(get_current_user)):
    if user["role"] != "admin":
        raise HTTPException(status_code=403, detail="Admins only")
    return user


@app.delete("/users/{user_id}")
def delete_user(user_id: int, admin = Depends(require_admin)):
    # FastAPI automatically ran: security → get_current_user → require_admin
    # only reaches here if user is an admin
    return {"deleted": user_id}

---
## PART 3 — Middleware & CORS

### What is Middleware?

Middleware wraps **every** request before it reaches your endpoint and every response before it leaves. It sits in between — hence "middle".

#### Restaurant analogy

Think of your API like a restaurant:

```
Customer (client)  →  Waiter (middleware)  →  Chef (endpoint)
```

The **waiter** doesn't cook the food — but every single order passes
through them. They can:
- Check if the customer has a reservation   (auth)
- Write down the order in a log             (logging)
- Start a timer when order is placed        (timing)
- Add a complimentary bread to every table  (global headers)
- Reject rude customers before they sit     (rate limiting)

The chef never thinks about any of this — they just cook.
That's exactly what middleware does.

---

### The Physical "Middle" — Why It's Called That

```
[ Client ]
    ↓  sends request
[ Middleware ]   ← sits in the MIDDLE
    ↓  forwards request
[ Your Endpoint ]
    ↓  sends response back
[ Middleware ]   ← catches it on the way out too
    ↓  forwards response
[ Client ]
```
Every request goes **through** middleware — in and out.
Your endpoint never sees what middleware intercepted or modified.

---

#### Real World: What Happens Without Middleware

Imagine you have 50 routes and you want to log every request:

```python
# ❌ without middleware — you repeat this in every single route
@app.get("/users")
def get_users():
    print("Request received at", time.time())   # repeated
    ...

@app.get("/items")
def get_items():
    print("Request received at", time.time())   # repeated
    ...

# 50 routes × same logging code = nightmare
```

With middleware — write it **once**, it applies everywhere:

```python
# ✅ with middleware — one place, covers all 50 routes
@app.middleware("http")
async def log_requests(request: Request, call_next):
    print(f"{request.method} {request.url} at {time.time()}")
    response = await call_next(request)   # all 50 routes handled here
    return response
```

---

#### The `call_next` Moment — This Is the Key

```python
@app.middleware("http")
async def my_middleware(request: Request, call_next):

    # ① everything here runs BEFORE your endpoint
    print("before")

    response = await call_next(request)   # ② your endpoint runs HERE
                                          #    middleware pauses and waits

    # ③ everything here runs AFTER your endpoint
    print("after")

    return response
```

> `call_next` is just a convention name — not a keyword.  
> FastAPI injects the actual function based on position.

| | Middleware | `Depends()` |
|--|------------|-------------|
| Applies to | Every route, always | Only opted-in routes |
| Best for | Logging, timing, IP blocking | Auth, DB sessions |
| Analogy | Security gate at building entrance | Keycard for a specific room |

### 3.1 — Custom Middleware

In [ ]:
import time
from fastapi import Request    # Request object gives you access to incoming request data
                               # method, URL, headers, body, client IP etc.

# @app.middleware - this is the decorator to register a middleware function
# "http" — tells FastAPI this middleware handles HTTP requests (explained below)
# every request to your app passes through this function automatically
@app.middleware("http")
async def timing_middleware(request: Request, call_next):
    # request  — the incoming request object (has method, url, headers etc.)
    # call_next — a function that passes the request to the next layer
    #             (either next middleware or the actual endpoint)
    # async def — required because call_next is awaitable


    # ── BEFORE the endpoint runs ──────────────────────────────────────────

    start = time.time()                              # record start time as float (e.g. 1718123456.123)
    print(f"Incoming: {request.method} {request.url}") # e.g. "Incoming: GET http://localhost:8000/users"


    # hand off to the next layer — execution PAUSES here until the endpoint finishes
    # whatever the endpoint returns becomes 'response'
    response = await call_next(request)


    # ── AFTER the endpoint runs ───────────────────────────────────────────

    duration = time.time() - start                   # end_time - start_time = how long it took
    response.headers["X-Process-Time"] = str(duration) # attach custom header to the response
                                                     # client can read this in browser devtools
    print(f"Completed in {duration:.3f}s")           # e.g. "Completed in 0.042s"

    return response    # send the response back to the client


### 3.2 — CORS (Cross-Origin Resource Sharing)

**The problem:** Your React frontend runs on `http://localhost:3000`.
Your FastAPI backend runs on `http://localhost:8000`. Different port =
different **origin** — the browser **blocks** this request by default.
This is a browser security rule, not a FastAPI rule.

```
React (localhost:3000)  →  ❌ blocked by browser  →  FastAPI (localhost:8000)
                         "Different origin, not allowed!"
```

CORS middleware tells the browser: *"it's okay, I trust requests from
these specific origins."*

```
React (localhost:3000)  →  ✅ allowed  →  FastAPI (localhost:8000)
                         "This origin is on the allow-list"
```

---

### What Counts as a Different Origin?

Origin = **scheme + domain + port**. If any one of these differs, it's
a different origin — even if everything else matches.

```
http://localhost:3000  vs  http://localhost:8000   ❌ different port
http://localhost:3000  vs  https://localhost:3000   ❌ different scheme
http://localhost:3000  vs  http://myapp.com:3000    ❌ different domain
http://localhost:3000  vs  http://localhost:3000    ✅ same origin
```

---

In [ ]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000",   # your React dev server
                   "https://myapp.com"],       # your production frontend
    allow_credentials=True,                    # allow cookies / auth headers
    allow_methods=["*"],                       # GET, POST, PUT, DELETE etc.
    allow_headers=["*"],                       # Authorization, Content-Type etc.
)

# For development only — allow ALL origins (never do this in production)
# allow_origins=["*"]

### 3.3 — Middleware Order Matters

Middleware stacks like layers of an onion — last added = outermost (runs first).

#### Two Ways to Add Middleware

```python
app.add_middleware(CORSMiddleware, ...)   # class-based — registered outside
@app.middleware("http")                   # function-based — registered inside
async def timing_middleware(...): ...
```

Both do the same job — just different syntax. **Order of registration still
decides nesting**: `add_middleware()` here goes outer (runs first),
`@app.middleware()` goes inner (runs second).

---

#### Execution Order

```
Request  →  CORS  →  timing  →  Endpoint  →  timing  →  CORS  →  Response
            (1)       (2)                     (4)        (5)
```

CORS checks the origin **first**, before timing even starts the clock.
On the way out, timing finishes its calculation **first**, then CORS
does its final checks before the response leaves.

---

```
Outer middleware = first gate in, last gate out.
Inner middleware = closer to the endpoint, runs in the middle.
```

In [ ]:
# Middleware stacks like layers of an onion — last added = outermost = runs first on request
# Execution order on every request:
#
#   Request  →  CORS  →  timing  →  endpoint  →  timing  →  CORS  →  Response
#               (1st)    (2nd)                    (2nd)     (1st)
#
# CORS added via add_middleware() → goes on the outside (runs first)
# timing added via @app.middleware() → goes on the inside (runs second)

# add_middleware() is another way to register middleware — used for class-based middleware
# @app.middleware("http") is used for function-based middleware
# both do the same job, just different syntax
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],    # allow requests from any domain (dev only — be specific in prod)
    allow_methods=["*"],    # allow all HTTP methods (GET, POST, PUT, DELETE etc.)
    allow_headers=["*"],    # allow all headers (Authorization, Content-Type etc.)
)

@app.middleware("http")
async def timing_middleware(request: Request, call_next):
    start = time.time()                    # record time before endpoint runs

    response = await call_next(request)    # endpoint runs here — pauses until it's done

    # calculate how long the endpoint took and attach it to the response header
    # client can see this in browser devtools under Response Headers
    response.headers["X-Process-Time"] = str(time.time() - start)

    return response


---
## PART 4 — Advanced Routing with APIRouter

### The Problem

As your API grows, putting all routes in `main.py` becomes a mess:
```python
# main.py — 500 lines, all mixed together
@app.get("/users")     ...
@app.get("/products")  ...
@app.get("/orders")    ...
@app.get("/payments") ...
```

`APIRouter` lets you split routes into separate files by feature, then plug them into the main app.

### 4.1 — Basic APIRouter

In [ ]:
# routers/users.py  ← separate file for user routes
from fastapi import APIRouter

router = APIRouter(
    prefix="/users",      # all routes here get /users prepended
    tags=["Users"],       # groups them under "Users" in Swagger docs
)

@router.get("/")          # actual path: GET /users/
def list_users():
    return [{"id": 1}]

@router.get("/{user_id}") # actual path: GET /users/42
def get_user(user_id: int):
    return {"id": user_id}

@router.post("/")         # actual path: POST /users/
def create_user():
    return {"created": True}

In [ ]:
# main.py  ← clean, just wires things together
from fastapi import FastAPI
# from routers import users, products, orders  ← in real project

app = FastAPI()

# app.include_router(users.router)           # plugs the users router in
# app.include_router(products.router)
# app.include_router(orders.router)

## 📁 Project Structure — APIRouter & include_router()

`main.py` only **wires things together** — no route definitions, no business logic.

```python
# main.py
from fastapi import FastAPI
from routers import users, products, orders

app = FastAPI()

app.include_router(users.router)
app.include_router(products.router)
app.include_router(orders.router)
```

```python
# routers/users.py — routes live here, not in main.py
from fastapi import APIRouter
router = APIRouter(prefix="/users", tags=["Users"])

@router.get("/")
def list_users(): ...
```

---

### Folder Layout

```
myapp/
├── main.py              ← only wires things, never defines routes
├── routers/
│   ├── users.py         ← /users routes
│   ├── products.py      ← /products routes
│   └── orders.py        ← /orders routes
```

---

### Why Split Like This?

```
❌ everything in main.py     → 500+ lines, hard to navigate
✅ routes split by feature   → each file has one responsibility
```

> 💡 `include_router()` plugs a router's routes (plus its `prefix`,
> `tags`, `dependencies`) into the main app. Reading `main.py` should
> tell you the entire API structure at a glance.

### 4.2 — API Versioning

When you need to release a v2 of your API without breaking v1 users.

In [ ]:
# routers/v1/users.py
from fastapi import APIRouter

router = APIRouter(prefix="/v1/users", tags=["Users v1"])  # convention: always name it 'router'

@router.get("/")         # GET /v1/users/
def list_users_v1():
    return [{"id": 1, "name": "Alice"}]


# ─────────────────────────────────────────────────────────────────────────────


# routers/v2/users.py
from fastapi import APIRouter

router = APIRouter(prefix="/v2/users", tags=["Users v2"])  # same name 'router', alias on import handles conflict

@router.get("/")         # GET /v2/users/
def list_users_v2():
    # v2 returns extra fields — v1 clients are unaffected, they still use /v1/users/
    return [{"id": 1, "name": "Alice", "email": "alice@example.com"}]


# ─────────────────────────────────────────────────────────────────────────────


# main.py
from fastapi import FastAPI
from routers.v1 import users as users_v1   # alias avoids name conflict — both files have 'router'
from routers.v2 import users as users_v2

app = FastAPI()

app.include_router(users_v1.router)        # registers all /v1/users/ routes
app.include_router(users_v2.router)        # registers all /v2/users/ routes
                                           # both versions live side by side
                                           # old clients use /v1/, new clients use /v2/


### 4.3 — Router-level Dependencies

Apply a dependency to **every route in a router** — no need to add
`Depends()` to each endpoint individually.

```python
def verify_admin(user = Depends(get_current_user)):
    if user["role"] != "admin":
        raise HTTPException(403, "Admins only")
    return user

admin_router = APIRouter(
    prefix="/admin",
    tags=["Admin"],
    dependencies=[Depends(verify_admin)]   # applied to ALL routes below
)

@admin_router.get("/stats")         # protected automatically
def get_stats():
    return {"total_users": 1000}

@admin_router.delete("/users/{id}") # protected automatically
def delete_user(id: int):
    return {"deleted": id}
```

---

### Without Router-level Dependencies

```python
# ❌ repeated in every single route
@admin_router.get("/stats", dependencies=[Depends(verify_admin)])
def get_stats(): ...

@admin_router.delete("/users/{id}", dependencies=[Depends(verify_admin)])
def delete_user(id: int): ...
```

```python
# ✅ written once, covers the whole router
admin_router = APIRouter(dependencies=[Depends(verify_admin)])
```

---

> 💡 Same idea as middleware, but scoped — **middleware** = every route
> in the app, **router-level dependency** = every route in *this* router only.

### 4.4 — Recommended Project Structure

```
myapp/
├── main.py              ← creates app, includes routers, adds middleware
├── routers/
│   ├── users.py         ← user routes
│   ├── products.py      ← product routes
│   └── orders.py        ← order routes
├── models/
│   └── schemas.py       ← Pydantic models (request/response shapes)
├── dependencies/
│   ├── auth.py          ← get_current_user, require_admin
│   └── database.py      ← get_db
└── core/
    └── config.py        ← settings (env vars, secrets)
```

`main.py` stays thin — it just wires everything together.

###  Real-World Middleware Use Cases

#### 1. Logging — most common
```python
@app.middleware("http")
async def log_middleware(request: Request, call_next):
    print(f"{request.method} {request.url.path} - from {request.client.host}")
    response = await call_next(request)
    print(f"Response: {response.status_code}")
    return response
```
Every company logs every request — who called what, when, from where.
Essential for debugging production issues.

#### 2. Timing / performance monitoring
```python
@app.middleware("http")
async def timing_middleware(request: Request, call_next):
    start = time.time()
    response = await call_next(request)
    duration = time.time() - start
    if duration > 2.0:
        print(f"SLOW REQUEST: {request.url.path} took {duration}s")
    response.headers["X-Process-Time"] = str(duration)
    return response
```
Flags slow endpoints that need optimization.

#### 3. CORS — almost every project
```python
app.add_middleware(CORSMiddleware, allow_origins=["https://myapp.com"])
```
Any project with a separate frontend (React, mobile app) needs this.

#### 4. App-level authentication
```python
OPEN_ROUTES = ["/", "/docs", "/login", "/register"]

@app.middleware("http")
async def auth_middleware(request: Request, call_next):
    if request.url.path not in OPEN_ROUTES:
        token = request.headers.get("Authorization")
        if not token:
            return JSONResponse(status_code=401, content={"detail": "Not authenticated"})
    return await call_next(request)
```
Blocks unauthenticated requests in one place instead of adding
`Depends(get_current_user)` everywhere.

#### 5. Rate limiting
```python
from collections import defaultdict
request_counts = defaultdict(int)

@app.middleware("http")
async def rate_limit_middleware(request: Request, call_next):
    ip = request.client.host
    request_counts[ip] += 1
    if request_counts[ip] > 100:
        return JSONResponse(status_code=429, content={"detail": "Too many requests"})
    return await call_next(request)
```
Stops one user from hammering your API.

#### 6. Maintenance mode
```python
@app.middleware("http")
async def maintenance_middleware(request: Request, call_next):
    if MAINTENANCE_MODE:
        return JSONResponse(status_code=503, content={"detail": "Under maintenance"})
    return await call_next(request)
```
Flip one variable — entire API goes down gracefully.

---

### When to Use Middleware vs `Depends()`

| Situation | Middleware? |
|---|---|
| Logic applies to **every** request | ✅ yes |
| Logic applies to **some** endpoints only | ❌ use `Depends()` |
| Need to inspect raw request/response | ✅ yes |
| Need DB session, current user object | ❌ use `Depends()` |

> 💡 Middleware = blanket rule for the whole app.
> `Depends()` = targeted logic for specific routes.

---
## Key Takeaways

| Concept | What to remember |
|---|---|
| **`Depends()`** | Injects reusable logic (auth, DB) into endpoints — write once, use everywhere |
| **`yield` in deps** | Code before yield = setup, code after = cleanup — perfect for DB sessions |
| **Chained deps** | Dependencies can depend on other dependencies — FastAPI resolves the chain |
| **Middleware** | Wraps every request automatically — use for logging, timing, global headers |
| **CORS** | Required when frontend and backend run on different origins (ports) |
| **APIRouter** | Splits routes into separate files by feature — keeps `main.py` clean |
| **Router deps** | Apply a dependency to all routes in a router in one line |

### What to run next
```bash
cd backend-and-systems/fastapi/examples
uvicorn dependencies_example:app --reload   # then open http://127.0.0.1:8000/docs
uvicorn middleware_cors_example:app --reload
uvicorn routing_example:app --reload
```